In [1]:
# ============================================================
# FRAUD DATA ANALYSIS USING NLP + K-MEANS
# Jupyter Notebook
# ============================================================

# ------------------------------------------------------------
# 1. Import libraries
# ------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import re
import warnings
warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 2. Load fraud_data.csv
# ------------------------------------------------------------

file_path = "fraud_data.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn names:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 3. Basic dataset information
# ------------------------------------------------------------

print("Dataset information:")
display(df.info())

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


# ------------------------------------------------------------
# 4. Identify a text column automatically
# ------------------------------------------------------------

text_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("Text/object columns:", text_columns)

if len(text_columns) == 0:
    raise ValueError(
        "No text column was found. Please add a text column containing "
        "transaction descriptions, messages, emails, etc."
    )

# Use the first text column by default
TEXT_COLUMN = text_columns[0]

print("Text column selected:", TEXT_COLUMN)


# ------------------------------------------------------------
# 5. Clean the text
# ------------------------------------------------------------

def clean_text(text):
    text = str(text).lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    
    # Remove email addresses
    text = re.sub(r"\S+@\S+", " ", text)
    
    # Remove numbers
    text = re.sub(r"\d+", " ", text)
    
    # Remove punctuation/special characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


df["clean_text"] = df[TEXT_COLUMN].fillna("").apply(clean_text)

display(df[[TEXT_COLUMN, "clean_text"]].head())


# ------------------------------------------------------------
# 6. Convert text into numerical features using TF-IDF
# ------------------------------------------------------------

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

X_text = vectorizer.fit_transform(df["clean_text"])

print("TF-IDF matrix shape:", X_text.shape)


# ------------------------------------------------------------
# 7. Find a suitable number of K-Means clusters
#    using the Elbow Method
# ------------------------------------------------------------

inertias = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    kmeans.fit(X_text)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Choosing K")
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# 8. Silhouette analysis
# ------------------------------------------------------------

silhouette_scores = []

for k in K_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels = kmeans.fit_predict(X_text)
    
    score = silhouette_score(X_text, labels)
    silhouette_scores.append(score)

plt.figure(figsize=(8, 5))
plt.plot(K_range, silhouette_scores, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score")
plt.grid(True)
plt.show()

best_k = K_range[np.argmax(silhouette_scores)]

print("Suggested number of clusters:", best_k)
print("Best silhouette score:", max(silhouette_scores))


# ------------------------------------------------------------
# 9. Apply K-Means
# ------------------------------------------------------------

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X_text)

print("\nCluster counts:")
display(df["cluster"].value_counts().sort_index())


# ------------------------------------------------------------
# 10. Examine the most important words in each cluster
# ------------------------------------------------------------

feature_names = vectorizer.get_feature_names_out()

for cluster_num in range(best_k):
    
    center = kmeans.cluster_centers_[cluster_num]
    
    top_indices = center.argsort()[-15:][::-1]
    top_words = feature_names[top_indices]
    
    print(f"\nCluster {cluster_num}")
    print("-" * 40)
    print(", ".join(top_words))


# ------------------------------------------------------------
# 11. Display sample transactions from each cluster
# ------------------------------------------------------------

for cluster_num in range(best_k):
    
    print("\n" + "=" * 70)
    print(f"CLUSTER {cluster_num}")
    print("=" * 70)
    
    cluster_data = df[df["cluster"] == cluster_num]
    
    display(
        cluster_data[
            [TEXT_COLUMN, "clean_text", "cluster"]
        ].head(10)
    )


# ------------------------------------------------------------
# 12. Visualise K-Means clusters using PCA
# ------------------------------------------------------------

# Convert sparse TF-IDF matrix to a lower-dimensional representation
pca = PCA(n_components=2, random_state=42)

X_dense = X_text.toarray()

X_pca = pca.fit_transform(X_dense)

plt.figure(figsize=(10, 7))

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df["cluster"],
    alpha=0.6
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("K-Means Clusters of Fraud Data")
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# 13. Add optional numerical features
# ------------------------------------------------------------

# Look for common transaction amount columns
possible_amount_columns = [
    "amount",
    "Amount",
    "transaction_amount",
    "TransactionAmount",
    "transaction_value"
]

amount_column = None

for column in possible_amount_columns:
    if column in df.columns:
        amount_column = column
        break

if amount_column:
    
    print("Amount column found:", amount_column)
    
    df[amount_column] = pd.to_numeric(
        df[amount_column],
        errors="coerce"
    )
    
    print("\nAverage transaction amount by cluster:")
    
    display(
        df.groupby("cluster")[amount_column]
        .agg(["count", "mean", "median", "min", "max"])
    )
    
    plt.figure(figsize=(8, 5))
    
    df.groupby("cluster")[amount_column].mean().plot(
        kind="bar"
    )
    
    plt.xlabel("Cluster")
    plt.ylabel("Average Transaction Amount")
    plt.title("Average Transaction Amount by Cluster")
    plt.xticks(rotation=0)
    plt.show()


# ------------------------------------------------------------
# 14. Cluster summary
# ------------------------------------------------------------

cluster_summary = (
    df.groupby("cluster")
      .size()
      .reset_index(name="Number_of_transactions")
)

cluster_summary["Percentage"] = (
    cluster_summary["Number_of_transactions"]
    / len(df) * 100
)

display(cluster_summary)


# ------------------------------------------------------------
# 15. Save the clustered dataset
# ------------------------------------------------------------

output_file = "fraud_data_clustered.csv"

df.to_csv(
    output_file,
    index=False
)

print(f"Clustered dataset saved as: {output_file}")


# ------------------------------------------------------------
# 16. Basic fraud-cluster interpretation
# ------------------------------------------------------------

print("\nFraud Cluster Interpretation")
print("=" * 50)

for cluster_num in range(best_k):
    
    cluster_size = (df["cluster"] == cluster_num).sum()
    
    center = kmeans.cluster_centers_[cluster_num]
    top_indices = center.argsort()[-10:][::-1]
    top_words = feature_names[top_indices]
    
    print(f"\nCluster {cluster_num}")
    print(f"Number of records: {cluster_size}")
    print("Important words:")
    print(", ".join(top_words))

Dataset shape: (21693, 30)


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,1.176563,0.323798,0.536927,1.047002,-0.368652,-0.728586,0.084678,-0.069246,-0.266389,0.155315,...,-0.109627,-0.341365,0.057845,0.499180,0.415211,-0.581949,0.015472,0.018065,4.67,0
1,0.681109,-3.934776,-3.801827,-1.147468,-0.735540,-0.501097,1.038865,-0.626979,-2.274423,1.527782,...,0.652202,0.272684,-0.982151,0.165900,0.360251,0.195321,-0.256273,0.056501,912.00,0
2,1.140729,0.453484,0.247010,2.383132,0.343287,0.432804,0.093380,0.173310,-0.808999,0.775436,...,-0.003802,0.058556,-0.121177,-0.304215,0.645893,0.122600,-0.012115,-0.005945,1.00,0
3,-1.107073,-3.298902,-0.184092,-1.795744,2.137564,-1.684992,-2.015606,-0.007181,-0.165760,0.869659,...,0.130648,0.329445,0.927656,-0.049560,-1.892866,-0.575431,0.266573,0.414184,62.10,0
4,-0.314818,0.866839,-0.124577,-0.627638,2.651762,3.428128,0.194637,0.670674,-0.442658,0.133499,...,-0.312774,-0.799494,-0.064488,0.953062,-0.429550,0.158225,0.076943,-0.015051,2.67,0



Column names:
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']
Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 21693 entries, 0 to 21692
Data columns (total 30 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   V1      21693 non-null  float64
 1   V2      21693 non-null  float64
 2   V3      21693 non-null  float64
 3   V4      21693 non-null  float64
 4   V5      21693 non-null  float64
 5   V6      21693 non-null  float64
 6   V7      21693 non-null  float64
 7   V8      21693 non-null  float64
 8   V9      21693 non-null  float64
 9   V10     21693 non-null  float64
 10  V11     21693 non-null  float64
 11  V12     21693 non-null  float64
 12  V13     21693 non-null  float64
 13  V14     21693 non-null  float64
 14  V15     21693 non-null  float64
 15  V16     21693 non-null  f

None


Missing values:


V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64


Duplicate rows: 143
Text/object columns: []


ValueError: No text column was found. Please add a text column containing transaction descriptions, messages, emails, etc.